# Parallel: train a Gaussian splat on a free GPU

This notebook turns a **training package** from Parallel into a trained Gaussian splat, using
nerfstudio's *Splatfacto* (the trainer Parallel's own GPU worker uses), and prepares the result for
**Import trained splat** in Parallel.

**What you need:** the training package `.zip` (Parallel: Captures, reconstruction job, *Train on a free GPU*).
**Runtime:** Runtime, Change runtime type, choose a **GPU** (T4 is enough).

**Rough time on a free T4 (an estimate, not a measurement):** a package of 150 to 240 frames trains for
15,000 iterations in about 25 to 45 minutes, plus 5 to 10 minutes of set-up the first time (nerfstudio
compiles its CUDA kernels). A free session can be cut off, so keep the page open and download the result
as soon as it is ready.

**What this notebook will not do:** it does not change the reconstruction's scale or orientation, and it
does not make the splat a measurement. A splat is how the place looks; Parallel keeps it separate from
the geometry used for simulation.

In [ ]:
# 1. Check the GPU
import subprocess, sys
smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if smi.returncode != 0:
    raise SystemExit("No GPU found. Use Runtime > Change runtime type and choose a GPU, then run again.")
GPU_NAME = smi.stdout.strip().splitlines()[0]
print("GPU:", GPU_NAME)

In [ ]:
# 2. Install nerfstudio (about 5 to 10 minutes). If pip reports a conflict, run Runtime > Restart session
#    once and run this notebook again from the top.
%pip install -q nerfstudio
import importlib.metadata as md_
print("nerfstudio", md_.version("nerfstudio"))

## 3. The helper code (tested in the Parallel repository; do not edit)

In [ ]:
import hashlib
import json
from pathlib import Path

import numpy as np

REQUIRED = ("x", "y", "z", "f_dc_0", "f_dc_1", "f_dc_2", "opacity", "scale_0", "scale_1", "scale_2",
            "rot_0", "rot_1", "rot_2", "rot_3")
_TYPES = {"float": "<f4", "float32": "<f4", "double": "<f8", "float64": "<f8", "uchar": "u1", "uint8": "u1",
          "int": "<i4", "int32": "<i4", "uint": "<u4", "uint32": "<u4"}
_NAMES = {"<f4": "float", "<f8": "double", "|u1": "uchar", "<i4": "int", "<u4": "uint"}


def sha256_file(path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(1 << 20):
            h.update(chunk)
    return h.hexdigest()


def verify_package(root) -> dict:
    """Check every file of an unzipped training package against its manifest. Returns the manifest."""
    root = Path(root)
    manifest = json.loads((root / "manifest.json").read_text())
    if manifest.get("format") != "parallel-training-package/1":
        raise ValueError(f"unknown package format {manifest.get('format')!r}")
    bad = [f["path"] for f in manifest["files"] if not (root / f["path"]).exists()
           or sha256_file(root / f["path"]) != f["sha256"]]
    if bad:
        raise ValueError(f"{len(bad)} package files are missing or changed (for example {bad[0]}); "
                         "upload the package again")
    manifest["manifest_sha256"] = sha256_file(root / "manifest.json")
    return manifest


def read_points3d(path) -> np.ndarray:
    rows = [ln.split()[1:4] for ln in Path(path).read_text().splitlines() if ln and not ln.startswith("#")]
    return np.array(rows, dtype=float).reshape(-1, 3)


def read_ply(path) -> tuple[np.ndarray, list[str]]:
    """Binary little-endian PLY with one vertex element of scalar properties -> structured array, comments."""
    with open(path, "rb") as f:
        head = b""
        while b"end_header" not in head:
            chunk = f.read(4096)
            if not chunk:
                raise ValueError("PLY header is not terminated")
            head += chunk
        end = head.index(b"end_header")
        body_start = head.index(b"\n", end) + 1
        f.seek(body_start)
        lines = head[:end].decode("ascii").splitlines()
        props, count, comments = [], 0, []
        for ln in lines[1:]:
            p = ln.split()
            if not p:
                continue
            if p[0] == "comment":
                comments.append(ln[8:])
            elif p[0] == "format" and p[1] != "binary_little_endian":
                raise ValueError(f"only binary little-endian PLY is supported, got {p[1]}")
            elif p[0] == "element" and p[1] == "vertex":
                count = int(p[2])
            elif p[0] == "property":
                props.append((p[2], _TYPES[p[1]]))
        data = np.fromfile(f, dtype=np.dtype(props), count=count)
    if len(data) != count:
        raise ValueError(f"truncated PLY: expected {count} Gaussians, read {len(data)}")
    missing = [n for n in REQUIRED if n not in data.dtype.names]
    if missing:
        raise ValueError(f"not a Gaussian splat PLY: missing {missing}")
    return data, comments


def write_ply(path, data: np.ndarray, comments: list[str] | None = None) -> None:
    header = ["ply", "format binary_little_endian 1.0"] + [f"comment {c}" for c in comments or []]
    header.append(f"element vertex {len(data)}")
    header += [f"property {_NAMES[data.dtype[n].str]} {n}" for n in data.dtype.names]
    header.append("end_header")
    with open(path, "wb") as f:
        f.write(("\n".join(header) + "\n").encode("ascii"))
        data.tofile(f)


def positions(data: np.ndarray) -> np.ndarray:
    return np.stack([data["x"], data["y"], data["z"]], axis=1).astype(np.float64)


def drop_non_finite(data: np.ndarray) -> tuple[np.ndarray, int]:
    ok = np.ones(len(data), dtype=bool)
    for n in REQUIRED:
        ok &= np.isfinite(data[n])
    return data[ok], int((~ok).sum())


def drop_view_dependent_colour(data: np.ndarray) -> np.ndarray:
    """Remove f_rest_* (view-dependent colour); about five times smaller, same base colour."""
    keep = [n for n in data.dtype.names if not n.startswith("f_rest_")]
    out = np.zeros(len(data), dtype=[(n, data.dtype[n]) for n in keep])
    for n in keep:
        out[n] = data[n]
    return out


def load_dataparser_transform(path) -> tuple[np.ndarray, np.ndarray, float] | None:
    """nerfstudio's dataparser_transforms.json -> (R, t, scale) with x_train = scale * (R x + t)."""
    p = Path(path)
    if not p.exists():
        return None
    j = json.loads(p.read_text())
    m = np.array(j["transform"], dtype=float)
    return m[:3, :3], m[:3, 3], float(j.get("scale", 1.0))


def is_identity(R: np.ndarray, t: np.ndarray, s: float, tol: float = 1e-6) -> bool:
    return np.allclose(R, np.eye(3), atol=tol) and np.allclose(t, 0, atol=tol) and abs(s - 1.0) < tol


def to_capture_frame(data: np.ndarray, R: np.ndarray, t: np.ndarray, s: float) -> np.ndarray:
    """Undo nerfstudio's orientation, centring and scaling so the splat is in COLMAP's frame again.

    Positions: x = R^T (x_train / s - t). Gaussian sizes scale with the positions (log-scales shift by
    -log s); orientations compose with R^T. View-dependent colour is not rotated, so it is dropped.
    """
    from scipy.spatial.transform import Rotation

    out = drop_view_dependent_colour(data) if any(n.startswith("f_rest_") for n in data.dtype.names) else data.copy()
    xyz = (positions(out) / s - t) @ R  # rows: R^T v == v @ R
    out["x"], out["y"], out["z"] = xyz[:, 0], xyz[:, 1], xyz[:, 2]
    for i in range(3):
        out[f"scale_{i}"] = out[f"scale_{i}"] - np.log(s)
    q = np.stack([out[f"rot_{i}"] for i in range(4)], axis=1).astype(np.float64)
    q = q / np.maximum(np.linalg.norm(q, axis=1, keepdims=True), 1e-12)
    new = (Rotation.from_matrix(R.T) * Rotation.from_quat(q[:, [1, 2, 3, 0]])).as_quat()  # xyzw
    for i, col in enumerate((3, 0, 1, 2)):
        out[f"rot_{i}"] = new[:, col]
    return out


def alignment_check(splat_xyz: np.ndarray, sfm_xyz: np.ndarray, max_relative_distance: float = 0.05,
                    spread_ratio_range: tuple[float, float] = (0.5, 2.0), sample: int = 20_000) -> dict:
    """The same statistics Parallel applies on import (see parallel_scene.splat.alignment_check)."""
    from scipy.spatial import cKDTree

    rng = np.random.default_rng(0)
    sfm = np.asarray(sfm_xyz, dtype=np.float64)
    splat = np.asarray(splat_xyz, dtype=np.float64)
    if len(splat) > 500_000:
        splat = splat[rng.choice(len(splat), 500_000, replace=False)]
    if len(sfm) > sample:
        sfm = sfm[rng.choice(len(sfm), sample, replace=False)]
    sfm_spread = float(np.median(np.linalg.norm(sfm - np.median(sfm, axis=0), axis=1)))
    splat_spread = float(np.median(np.linalg.norm(splat - np.median(splat, axis=0), axis=1)))
    dist, _ = cKDTree(splat).query(sfm)
    median_rel = float(np.median(dist)) / sfm_spread
    ratio = splat_spread / sfm_spread
    ok = median_rel <= max_relative_distance and spread_ratio_range[0] <= ratio <= spread_ratio_range[1]
    return {"passed": bool(ok), "median_distance_fraction_of_spread": round(median_rel, 5),
            "spread_ratio_splat_to_sparse": round(ratio, 4)}

In [ ]:
# 4. Upload the training package (.zip). In Colab, a file chooser appears; on Kaggle or Drive,
#    set PACKAGE_ZIP to the file's path instead, for example "/kaggle/input/my-data/parallel-training-xxxx.zip".
import os, shutil, zipfile
from pathlib import Path

PACKAGE_ZIP = None  # e.g. "/content/drive/MyDrive/parallel-training-xxxx.zip"
if PACKAGE_ZIP is None:
    from google.colab import files
    up = files.upload()
    PACKAGE_ZIP = next(iter(up))
ROOT = Path("/content/capture")
shutil.rmtree(ROOT, ignore_errors=True)
with zipfile.ZipFile(PACKAGE_ZIP) as z:
    z.extractall(ROOT)
manifest = verify_package(ROOT)
print(f"Package for reconstruction job {manifest['job_id']}: {manifest['frames']} frames, "
      f"camera model {manifest['colmap_camera_model']}. All {len(manifest['files'])} files match the manifest.")

In [ ]:
# 5. Settings. 15,000 iterations is a good balance for a free GPU; raise to 30,000 for more detail if the
#    session allows. DOWNSCALE 2 halves the image size, which saves memory and time.
ITERATIONS = 15000
DOWNSCALE = 2
KEEP_VIEW_DEPENDENT_COLOUR = False  # True makes the file about 5 times larger for a small gain in realism

In [ ]:
# 6. Train. The orientation, centring and scale options keep the splat in the reconstruction's own frame.
import datetime, time
STARTED = datetime.datetime.now(datetime.timezone.utc).isoformat()
t0 = time.time()
CMD = [
    "ns-train", "splatfacto", "--data", str(ROOT), "--output-dir", "/content/outputs",
    "--max-num-iterations", str(ITERATIONS), "--vis", "tensorboard",
    "colmap", "--colmap-path", "colmap/sparse/0", "--images-path", "images",
    "--downscale-factor", str(DOWNSCALE), "--eval-mode", "all",
    "--orientation-method", "none", "--center-method", "none", "--auto-scale-poses", "False",
    "--assume-colmap-world-coordinate-convention", "False",
]
print(" ".join(CMD))
subprocess.run(CMD, check=True)
FINISHED = datetime.datetime.now(datetime.timezone.utc).isoformat()
print(f"Training took {(time.time() - t0) / 60:.1f} minutes")

In [ ]:
# 7. Export, then bring the splat back into the reconstruction's frame and check it
import glob
cfg = sorted(glob.glob("/content/outputs/**/config.yml", recursive=True), key=os.path.getmtime)[-1]
EXPORT = Path("/content/export")
shutil.rmtree(EXPORT, ignore_errors=True)
subprocess.run(["ns-export", "gaussian-splat", "--load-config", cfg, "--output-dir", str(EXPORT)], check=True)
raw = sorted(EXPORT.glob("*.ply"))[0]
data, _ = read_ply(raw)
data, dropped = drop_non_finite(data)
tr = load_dataparser_transform(Path(cfg).parent / "dataparser_transforms.json")
if tr is not None and not is_identity(*tr):
    print(f"nerfstudio changed the frame (scale {tr[2]:.4f}); converting back to the reconstruction's frame")
    data = to_capture_frame(data, *tr)
elif not KEEP_VIEW_DEPENDENT_COLOUR:
    data = drop_view_dependent_colour(data)
check = alignment_check(positions(data), read_points3d(ROOT / "colmap/sparse/0/points3D.txt"))
print(f"{len(data):,} Gaussians ({dropped} invalid removed).")
print("Alignment with the reconstruction:", check)
if not check["passed"]:
    raise SystemExit("The splat does not line up with the reconstruction, and Parallel would refuse it. "
                     "Check that the training command in step 6 ran unchanged and that the package was not edited.")
print("PASS: the splat lines up with the reconstruction.")

In [ ]:
# 8. Save and download: upload BOTH files to Parallel with "Import trained splat"
import json
OUTDIR = Path("/content/parallel_splat")
shutil.rmtree(OUTDIR, ignore_errors=True)
OUTDIR.mkdir()
write_ply(OUTDIR / "splat.ply", data, ["Gaussian splat trained with nerfstudio Splatfacto for a Parallel training package"])
info = {
    "format": "parallel-splat-training-info/1",
    "job_id": manifest["job_id"], "package_manifest_sha256": manifest["manifest_sha256"],
    "trainer": "nerfstudio splatfacto", "trainer_version": md_.version("nerfstudio"), "gpu": GPU_NAME,
    "iterations": ITERATIONS, "downscale_factor": DOWNSCALE, "started_at": STARTED, "finished_at": FINISHED,
    "view_dependent_colour": bool(KEEP_VIEW_DEPENDENT_COLOUR), "command": " ".join(CMD),
    "alignment_check": check,
}
(OUTDIR / "training_info.json").write_text(json.dumps(info, indent=1))
print("splat.ply:", round((OUTDIR / "splat.ply").stat().st_size / 1e6, 1), "MB")
try:
    from google.colab import files
    files.download(str(OUTDIR / "splat.ply"))
    files.download(str(OUTDIR / "training_info.json"))
except ImportError:
    print("Download both files from", OUTDIR)

## Back in Parallel

Open the project's **Captures** page, find the reconstruction job, choose **Import trained splat**, and select
`splat.ply` together with `training_info.json`. Parallel checks the file, records who imported it and from what
training, and adds it to the scene as a new revision. Your original untrained visual stays in the scene.